In [1]:
# %% [markdown]
# # 04_full_experiment.ipynb
# Full v2 architecture experiment: 60 questions x 5 iterations = 600
# API calls. Mirrors credit_rating_benchmark_experiment.ipynb's
# structure (resume support, per-question progress logging) but
# calls query_semantic_layer_v2 and records the compile_error vs
# execution_error taxonomy that v1 conflated into one field.
#
# Requires: utils.py (with query_semantic_layer_v2), semantic_compiler.py
# (round-3 fixed version), financial_semantic_layer.yaml (with
# monthly_payment dimension), credit_rating_questions_all.json, and a
# working DB_PATH, all in this directory. OPENAI_API_KEY in .env.
#
# This does NOT touch v1's results_phase3_v2.csv -- saves to a
# separate file so both remain independently available for the
# manuscript's comparison tables.

# %%
print("SCRIPT VERSION: 2026-07-15-v1")

import json
import pandas as pd
import time
from utils import query_semantic_layer_v2, evaluate, DB_PATH

print(f"DB_PATH in use: {DB_PATH}")

with open('credit_rating_questions_all.json', 'r', encoding='utf-8') as f:
    questions = json.load(f)

df_q = pd.DataFrame(questions)
total = len(df_q)
print(f"Total questions: {total}")
if total != 60:
    print(f"!! WARNING: expected 60 questions, found {total}. Check the question file.")

N_ITER = 5
RESULTS_PATH = './results_phase3_v2_arch.csv'

print(f"Iterations per question: {N_ITER}")
print(f"Total API calls if run from scratch: {total * N_ITER}")

# %% [markdown]
# ## 1. Resume support (same pattern as v1's
#     credit_rating_benchmark_experiment.ipynb)

# %%
try:
    df_saved = pd.read_csv(RESULTS_PATH)
    results = df_saved.to_dict('records')
    done_ids = set(
        q_id for q_id, grp in df_saved.groupby('question_id')
        if grp['iteration'].nunique() >= N_ITER
    )
    print(f"Resuming — completed: {len(done_ids)} / {total}")
except FileNotFoundError:
    results = []
    done_ids = set()
    print("Starting fresh.")

remaining = total - len(done_ids)
print(f"Remaining questions: {remaining}")
print(f"Remaining API calls: {remaining * N_ITER}")

# %% [markdown]
# ## 2. Run the full experiment

# %%
print("\n" + "=" * 70)
print("RUNNING FULL v2 EXPERIMENT")
print("=" * 70)

for i, row in df_q.iterrows():
    q_id = row['question_id']
    question = row['question']
    evidence = row['evidence']
    gold_sql = row['SQL']
    difficulty = row['difficulty']
    category = row['category']

    if q_id in done_ids:
        print(f"[{i+1:2d}/{total}] {q_id} — skipped (done)")
        continue

    iter_correct = 0
    iter_compile_err = 0
    iter_exec_err = 0

    for iteration in range(N_ITER):
        try:
            res = query_semantic_layer_v2(question, evidence)
        except Exception as e:
            results.append({
                'question_id': q_id, 'difficulty': difficulty, 'category': category,
                'iteration': iteration, 'is_correct': False,
                'compile_error': None, 'execution_error': f"API call failed: {e}",
                'concept_sql': None, 'compiled_sql': None, 'gold_sql': gold_sql,
                'question': question, 'latency_ms': None, 'prompt_chars': None,
                'substitutions': None,
            })
            iter_exec_err += 1
            continue

        if res['compile_error']:
            results.append({
                'question_id': q_id, 'difficulty': difficulty, 'category': category,
                'iteration': iteration, 'is_correct': False,
                'compile_error': res['compile_error'], 'execution_error': None,
                'concept_sql': res['concept_sql'], 'compiled_sql': None, 'gold_sql': gold_sql,
                'question': question, 'latency_ms': res['latency_ms'],
                'prompt_chars': res['prompt_chars'],
                'substitutions': ' | '.join(res['substitutions']) if res['substitutions'] else None,
            })
            iter_compile_err += 1
            continue

        ev = evaluate(res['sql'], gold_sql)
        correct = bool(ev['is_correct']) if ev['is_correct'] is not None else False
        if correct:
            iter_correct += 1
        if ev.get('error'):
            iter_exec_err += 1

        results.append({
            'question_id': q_id, 'difficulty': difficulty, 'category': category,
            'iteration': iteration, 'is_correct': correct,
            'compile_error': None, 'execution_error': ev.get('error'),
            'concept_sql': res['concept_sql'], 'compiled_sql': res['sql'], 'gold_sql': gold_sql,
            'question': question, 'latency_ms': res['latency_ms'],
            'prompt_chars': res['prompt_chars'],
            'substitutions': ' | '.join(res['substitutions']) if res['substitutions'] else None,
        })

        time.sleep(0.3)

    # checkpoint after every question, same as v1's pattern
    pd.DataFrame(results).to_csv(RESULTS_PATH, index=False, encoding='utf-8-sig')

    df_tmp = pd.DataFrame(results)
    running_acc = df_tmp['is_correct'].mean()
    print(f"[{i+1:2d}/{total}] {q_id} ({difficulty[:3]}/{category[:8]}) "
          f"correct={iter_correct}/{N_ITER} compile_err={iter_compile_err}/{N_ITER} "
          f"exec_err={iter_exec_err}/{N_ITER} | Running accuracy: {running_acc:.1%}")

print("=" * 70)
print(f"Experiment complete. Saved: {RESULTS_PATH}")
print("=" * 70)

# %% [markdown]
# ## 3. Final summary

# %%
df_final = pd.read_csv(RESULTS_PATH)

print(f"Total records: {len(df_final)}")
print(f"Questions: {df_final['question_id'].nunique()}")
print(f"Iterations: {df_final['iteration'].nunique()}")

print(f"\nOverall accuracy: {df_final['is_correct'].mean():.1%}")

n_compile_err = df_final['compile_error'].notna().sum()
n_exec_err = df_final['execution_error'].notna().sum()
n_correct = df_final['is_correct'].sum()
n_wrong_result = len(df_final) - n_compile_err - n_exec_err - n_correct

print(f"\n--- Two-stage failure taxonomy (full run) ---")
print(f"  Correct         : {n_correct} ({n_correct/len(df_final):.1%})")
print(f"  Compile errors  : {n_compile_err} ({n_compile_err/len(df_final):.1%})")
print(f"  Execution errors: {n_exec_err} ({n_exec_err/len(df_final):.1%})")
print(f"  Wrong result    : {n_wrong_result} ({n_wrong_result/len(df_final):.1%})")

print(f"\n--- Accuracy by difficulty ---")
print(df_final.groupby('difficulty')['is_correct'].agg(['mean', 'count']))

print(f"\n--- Accuracy by category ---")
print(df_final.groupby('category')['is_correct'].agg(['mean', 'count']))

print(f"\n--- Latency ---")
print(df_final.groupby('question_id')['latency_ms'].mean().describe())

print("""

NEXT STEPS:
  - Compare this accuracy against v1's results_phase3_v2.csv (38.3%
    SL / 39.0% SQL) to see whether the true information-hiding
    architecture maintains comparable accuracy.
  - Re-run the exposure verification (get_schema_context_sl_v2 +
    measure_exposure) to confirm 0/15 holds at full scale, not just
    in the pilot sample.
  - Feed this file into the same question-level statistical
    reanalysis (paired t-test, Wilcoxon, question-level bootstrap,
    TOST equivalence) built for addressing R1 Concerns #2/#3, applied
    now to v2 instead of v1.
  - Inspect any remaining wrong_result cases by category to determine
    which are addressable (few-shot/YAML gaps) vs. which reflect a
    genuine measure-design limitation to document in the manuscript
    (e.g. CR15's avg_amount measure not supporting the monthly-
    resegmentation logic some gold queries require).
""")

SCRIPT VERSION: 2026-07-15-v1
DB_PATH in use: ./dev/dev_20240627/dev_databases/dev_databases/financial/financial.sqlite
Total questions: 60
Iterations per question: 5
Total API calls if run from scratch: 300
Starting fresh.
Remaining questions: 60
Remaining API calls: 300

RUNNING FULL v2 EXPERIMENT
[ 1/60] CR01 (sim/default_) correct=1/5 compile_err=4/5 exec_err=0/5 | Running accuracy: 20.0%
[ 2/60] CR02 (sim/default_) correct=5/5 compile_err=0/5 exec_err=0/5 | Running accuracy: 60.0%
[ 3/60] CR03 (sim/default_) correct=5/5 compile_err=0/5 exec_err=0/5 | Running accuracy: 73.3%
[ 4/60] CR04 (sim/loan_por) correct=5/5 compile_err=0/5 exec_err=0/5 | Running accuracy: 80.0%
[ 5/60] CR05 (sim/loan_por) correct=5/5 compile_err=0/5 exec_err=0/5 | Running accuracy: 84.0%
[ 6/60] CR06 (sim/client_p) correct=0/5 compile_err=2/5 exec_err=3/5 | Running accuracy: 70.0%
[ 7/60] CR07 (sim/regional) correct=5/5 compile_err=0/5 exec_err=0/5 | Running accuracy: 74.3%
[ 8/60] CR08 (sim/transact) correc